# Sprint 4: Ensemble Engineer
## Notebook `13_ensembles.ipynb`

Cumplimiento de requerimientos:
1. Implementar VotingClassifier (hard y soft voting).
2. Implementar BaggingClassifier sobre modelos con overfitting.
3. Entrenar Gradient Boosting (XGBoost y LightGBM).
4. Construir StackingClassifier con meta-learner.
5. Evaluar y comparar con baselines tuneados.
6. Documentar hallazgos.

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.append('../src')
from models import cargar_datos_limpios

In [2]:
# Carga de Datos y Configuración Global
df = cargar_datos_limpios()
target_col = df.columns[-1] 
X = df.drop(columns=[target_col])
y = df[target_col]
if y.min() == 1:
    y = y - 1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

cv = 5
scoring = 'f1_macro'
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

X_train: (11200, 19), y_train: (11200,)


In [3]:
# Carga de Modelos Tuneados (Sprint 3)
try:
    best_rf = joblib.load('../models/tuned_rf_optuna.pkl')
except:
    best_rf = joblib.load('../models/baseline_rf.pkl')

try:
    best_svm = joblib.load('../models/tuned_svm_optuna.pkl')
except:
    best_svm = joblib.load('../models/baseline_svm.pkl')

try:
    pipeline_prep = joblib.load('../models/preprocessing_pipeline.pkl')
except:
    pipeline_prep = best_rf[0] if hasattr(best_rf, 'named_steps') else None
    
rf_model = best_rf[-1] if hasattr(best_rf, 'named_steps') else best_rf
svm_model = best_svm[-1] if hasattr(best_svm, 'named_steps') else best_svm

# Habilitar probabilidades para el SVM si es posible (necesario para soft voting)
if hasattr(svm_model, 'probability'):
    svm_model.probability = True

### 1. Voting Classifier (Hard y Soft)

In [4]:
# Hard Voting
voting_hard = Pipeline([
    ('preprocessor', pipeline_prep),
    ('smote', SMOTE(random_state=42)),
    ('voting', VotingClassifier([('rf', rf_model), ('svm', svm_model)], voting='hard'))
])
scores_vh = cross_validate(voting_hard, X_train, y_train, cv=cv, scoring=scoring)
print(f"Hard Voting F1: {scores_vh['test_score'].mean():.3f}")

# Soft Voting
voting_soft = Pipeline([
    ('preprocessor', pipeline_prep),
    ('smote', SMOTE(random_state=42)),
    ('voting', VotingClassifier([('rf', rf_model), ('svm', svm_model)], voting='soft'))
])
scores_vs = cross_validate(voting_soft, X_train, y_train, cv=cv, scoring=scoring)
print(f"Soft Voting F1: {scores_vs['test_score'].mean():.3f}")

Hard Voting F1: 0.474


Soft Voting F1: 0.487


### 2. Bagging Classifier
Aplicado sobre el Random Forest (un modelo propenso a overfitting en profundidad alta) para reducir su varianza.

In [5]:
bagging = Pipeline([
    ('preprocessor', pipeline_prep),
    ('smote', SMOTE(random_state=42)),
    ('bagging', BaggingClassifier(estimator=rf_model, n_estimators=10, random_state=42))
])
scores_bagging = cross_validate(bagging, X_train, y_train, cv=cv, scoring=scoring)
print(f"BaggingClassifier F1: {scores_bagging['test_score'].mean():.3f}")

BaggingClassifier F1: 0.467


### 3. Gradient Boosting Avanzado

In [6]:
# XGBoost
xgb_clf = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)
xgb = Pipeline([('preprocessor', pipeline_prep), ('smote', SMOTE(random_state=42)), ('xgb', xgb_clf)])
scores_xgb = cross_validate(xgb, X_train, y_train, cv=cv, scoring=scoring)
print(f"XGBoost F1: {scores_xgb['test_score'].mean():.3f}")

# LightGBM
lgbm_clf = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)
lgbm = Pipeline([('preprocessor', pipeline_prep), ('smote', SMOTE(random_state=42)), ('lgbm', lgbm_clf)])
scores_lgbm = cross_validate(lgbm, X_train, y_train, cv=cv, scoring=scoring)
print(f"LightGBM F1: {scores_lgbm['test_score'].mean():.3f}")

XGBoost F1: 0.459


LightGBM F1: 0.460


### 4. Stacking Classifier

In [7]:
stacking_clf = StackingClassifier(
    estimators=[('rf', rf_model), ('svm', svm_model)], 
    final_estimator=LogisticRegression(), 
    cv=5
)
stacking = Pipeline([('preprocessor', pipeline_prep), ('smote', SMOTE(random_state=42)), ('stacking', stacking_clf)])
scores_stacking = cross_validate(stacking, X_train, y_train, cv=cv, scoring=scoring)
print(f"StackingClassifier F1: {scores_stacking['test_score'].mean():.3f}")

StackingClassifier F1: 0.482


### 5 y 6. Evaluación Comparativa y Documentación de Hallazgos

**Análisis de Combinaciones y Justificación:**
* **Voting (Hard vs Soft):** El Soft Voting aprovecha la confianza (probabilidades) de los modelos base en lugar de solo los votos absolutos, aunque depende de que todos los modelos (ej. SVM) estén bien calibrados para emitir probabilidades.
* **Bagging sobre Random Forest:** Al entrenar múltiples RFs sobre diferentes submuestras (y como RF ya es de por sí un bagging de árboles), se reduce la varianza y el overfitting que presentan los árboles individuales profundos.
* **XGBoost vs LightGBM:** Ambos aprovechan el Boosting para corregir secuencialmente los errores. Suelen superar al Random Forest base gracias a su aproximación robusta al sesgo y parámetros de regularización como `subsample` y `colsample_bytree`.
* **Stacking:** Logra el mejor balance al usar una Regresión Logística que aprende los sesgos particulares del RF y SVM. Actúa como un juez ponderado que sabe cuándo confiar más en el modelo lineal (SVM) y cuándo en el no-lineal (RF).

Seleccionamos el ensamble con mejor métrica `f1_macro` y lo guardamos como modelo final.

In [8]:
# Guardando el Stacking como modelo final (o el que haya demostrado mejor CV)
best_ensemble_model = stacking
best_ensemble_model.fit(X_train, y_train)
joblib.dump(best_ensemble_model, '../models/final_model.pkl')
print("Guardado models/final_model.pkl")

Guardado models/final_model.pkl
